# iRage AlgoArena — Short-Horizon Return Prediction | v6

## v5 -> v6 Diagnostic-Driven Upgrades

v5 output revealed:
- LGB stopped at iter 22-49 = SEVERE UNDERFITTING (LR=0.05 too high)
- OOF R2=0.0005 is the REAL honest signal (v4 0.030/0.055 were GroupKFold-leaked)
- 916 features but model used ~682 total splits = >900 features never touched

| Change | v5 | v6 | Justification |
|--------|----|----|---------------|
| LGB LR | 0.05 (stops @22 iter) | 0.01 | 22 iter = underfitting; need 500-2000 |
| LGB num_leaves | 31 | 63 | 661k rows with 916 feats needs more capacity |
| LGB min_child_samples | 300 | 150 | Too restrictive for low-signal regime |
| LGB early_stop | 100 | 300 | Low-SNR needs patience to find signal |
| LGB objective | regression/MSE | huber (alpha=0.9) | Kurtosis=48 -> MSE blown by outliers |
| Feature selection | All 916 | Top 300 by gain | Remove noise, improve split quality |
| New FE: intra-family z-score | None | Added | Cross-sectional alpha isolation (Fama-French) |
| New FE: Price sqrt-impact | None | Added | Almgren et al. square-root impact model |
| CatBoost loss | RMSE | Huber:delta=0.05 | Same outlier-robustness reasoning |
| 3rd model: LGB quantile | None | Added (alpha=0.5) | Median regression = max outlier robustness |
| Feature normalization | None | z-score (train stats) | Different feature scales waste histogram bins |

## GPT Advice Audit (REJECT: group normalization)
GPT suggested normalizing by CV_GROUP. REJECTED because:
CV_GROUP is ONLY in train.parquet. It is absent from test.parquet.
Applying group normalization to train without matching groups on test
creates a train/test distribution mismatch -> guaranteed to HURT test R2.
The competition explicitly states CV_GROUP is for CV splits only, not features.

## Runtime Budget (30 min)
| Phase | Est. Time |
|---|---|
| Load + FE + feature selection | ~90s |
| LGB regression: 5 folds (LR=0.01, early_stop=300, ~1500 iter) | ~600s |
| LGB quantile: 5 folds (same params) | ~600s |
| CatBoost: 5 folds x 1000 iter LR=0.04 | ~1150s |
| Ensemble + save | ~20s |
| Total | ~1800s = 30 min (with dynamic guards) |

In [16]:
# CELL 1 - IMPORTS
import os, gc, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize

import lightgbm as lgb
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import r2_score

try:
    from catboost import CatBoostRegressor, Pool
    CATBOOST_AVAILABLE = True
    print('CatBoost OK')
except ImportError:
    CATBOOST_AVAILABLE = False
    print('CatBoost not installed')

SEED = 23
np.random.seed(SEED)
NOTEBOOK_START = time.time()
print(f'LightGBM {lgb.__version__}')
print('Imports OK')

CatBoost OK
LightGBM 4.6.0
Imports OK


In [17]:
# CELL 2 - CONFIG
#
# KEY v6 REASONING:
#
# 1. LGB UNDERFITTING FIX
#    v5 stopped at iter 22-49 with LR=0.05.
#    22 iterations * 31 leaves = 682 total splits across 916 features.
#    That is less than 1 split per feature. Model never explored the data.
#    Fix: LR=0.01, num_leaves=63. Expected convergence: 500-2000 iter.
#    With early_stop=300: model gets real chance to find signal.
#
# 2. HUBER LOSS
#    TARGET kurtosis=48. In a normal distribution kurtosis=3.
#    kurtosis=48 means tails are ~16x heavier than Gaussian.
#    MSE loss: gradient proportional to (y_pred - y_true)^2.
#    A single outlier at 10x average magnitude gets 100x gradient weight.
#    With 2% extreme rows, ~13k rows dominate gradient -> model fits outliers.
#    Huber: linear loss for |residual| > delta, quadratic below delta.
#    delta=0.05 (roughly 99th pct of clipped target std=0.031).
#    This is the statistically correct loss for fat-tailed targets.
#
# 3. FEATURE SELECTION
#    Bias-variance tradeoff: with N features and fixed model capacity C,
#    each feature gets C/N "budget". Removing noise features gives real
#    signal features more budget. LGB with 916 features and 22 iter:
#    each feature got <1 split on average. With 300 features: ~2+ splits.
#
# 4. INTRA-FAMILY Z-SCORE
#    For a row at time t: z_fi = (f_i - mean_family) / std_family
#    This is cross-sectional factor neutralization.
#    In factor investing: removes common beta, isolates idiosyncratic alpha.
#    An instrument significantly above/below its family at time t:
#    carries idiosyncratic signal about its own future return.
#    Row-wise computation -> NO test leakage.
#
# 5. QUANTILE LGB (alpha=0.5 = median regression)
#    Median regression minimizes |residual| instead of residual^2.
#    Provably optimal under Laplace noise (which fat-tailed returns approximate).
#    Different gradient signal than MSE/Huber -> different tree structure.
#    Ensemble of regression + quantile reduces variance via error decorrelation.

class CFG:
    if os.path.exists('/kaggle/input'):
        DATA_DIR = Path('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage')
    else:
        DATA_DIR = Path(r'F:\#Project\iRAGE_Hackathon\algo4\kaggle\input\competitions\short-horizon-return-prediction-challenge-by-i-rage')
    TRAIN_PATH = DATA_DIR / 'train.parquet'
    TEST_PATH  = DATA_DIR / 'test.parquet'
    SAMPLE_SUB = DATA_DIR / 'sample_submission.csv'
    if os.path.exists('/kaggle/working'):
        OUTPUT_DIR = Path('/kaggle/working')
    else:
        OUTPUT_DIR = Path('./output')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    N_FOLDS = 5
    SEED    = 23

    # P1/P99: retains more tail signal than P2/P98.
    # Training-only clip. Test predictions unconstrained.
    TARGET_CLIP = True
    CLIP_LOWER  = 1
    CLIP_UPPER  = 99

    # Feature selection: top K features by LGB gain importance
    # (measured on a single fold before full CV)
    TOP_K_FEATURES = 300

    # ── LGB REGRESSION (Huber loss) ────────────────────────────
    # LR=0.01: slow enough to find real signal in low-SNR regime
    # num_leaves=63: 2x v5, appropriate for 661k rows
    # min_child_samples=150: reduced from 300 (allows finer splits)
    # early_stop=300: patience for weak signal
    # objective=huber, alpha=0.9: robust to kurtosis=48 target
    #   alpha=0.9 means: Huber transition at 90th percentile of residuals
    #   -> most rows get quadratic loss, extreme 10% get linear
    LGB_PARAMS = {
        'objective':         'huber',
        'alpha':             0.9,
        'metric':            'huber',
        'learning_rate':     0.01,
        'num_leaves':        63,
        'max_depth':         -1,
        'min_child_samples': 150,
        'feature_fraction':  0.6,
        'bagging_fraction':  0.7,
        'bagging_freq':      1,
        'lambda_l1':         0.5,
        'lambda_l2':         5.0,
        'min_gain_to_split': 0.001,
        'path_smoothing':    0.1,
        'verbose':           -1,
        'n_jobs':            -1,
        'seed':              SEED,
    }
    LGB_NUM_ROUNDS = 3000
    LGB_EARLY_STOP = 300

    # ── LGB QUANTILE (median regression, alpha=0.5) ────────────
    # Minimizes sum|y - y_hat| (L1 loss = median)
    # Provably robust: breakdown point = 50% (half data can be outliers)
    # Same tree structure budget but different gradient signal
    # -> decorrelated errors vs Huber model -> better ensemble
    LGB_Q_PARAMS = {
        'objective':         'quantile',
        'alpha':             0.5,
        'metric':            'quantile',
        'learning_rate':     0.01,
        'num_leaves':        63,
        'max_depth':         -1,
        'min_child_samples': 150,
        'feature_fraction':  0.6,
        'bagging_fraction':  0.7,
        'bagging_freq':      1,
        'lambda_l1':         0.5,
        'lambda_l2':         5.0,
        'min_gain_to_split': 0.001,
        'verbose':           -1,
        'n_jobs':            -1,
        'seed':              SEED + 1,
    }
    LGB_Q_NUM_ROUNDS = 3000
    LGB_Q_EARLY_STOP = 300

    # ── CATBOOST (Huber loss) ───────────────────────────────────
    # CatBoost structural advantages (ordered boosting, oblivious trees)
    # are MORE important after fixing CV leakage.
    # loss_function=Huber:delta=0.05
    #   delta=0.05 = ~1.6 std deviations of clipped target (std=0.031)
    #   -> residuals within 1.6 std get quadratic loss (95% of rows)
    #   -> extreme residuals get linear loss
    # iterations=1000 to fit 30-min budget with 3 model types
    CAT_PARAMS = dict(
        iterations=1000,
        learning_rate=0.04,
        depth=5,
        l2_leaf_reg=10.0,
        min_data_in_leaf=150,
        border_count=128,
        loss_function='Huber:delta=0.05',
        eval_metric='RMSE',
        random_seed=SEED,
        od_type='Iter',
        od_wait=100,
        verbose=200,
        task_type='CPU',
        thread_count=-1,
        allow_writing_files=False,
    )

    RUNTIME_LIMIT_S   = 1620   # 27 min hard cap
    CAT_SECS_PER_ITER = 0.231  # from v4 timing (1154s / 5000 iter)

print('Config OK')

Config OK


In [18]:
# CELL 3 - DATA LOADING
print('Loading data...')
t0 = time.time()

train = pd.read_parquet(CFG.TRAIN_PATH)
test  = pd.read_parquet(CFG.TEST_PATH)

print(f'Train: {train.shape} | Test: {test.shape}')
print(f'Loaded in {time.time()-t0:.1f}s')

train_ids = train['ID'].values.copy()
test_ids  = test['ID'].values.copy()
y_raw     = train['TARGET'].copy()

if 'CV_GROUP' in train.columns:
    cv_groups    = train['CV_GROUP'].values.copy()
    USE_GROUP_CV = True
    n_groups     = len(np.unique(cv_groups))
    print(f'CV_GROUP: {n_groups} groups -> GroupKFold')
    train.drop(columns=['ID', 'TARGET', 'CV_GROUP'], inplace=True)
else:
    cv_groups    = None
    USE_GROUP_CV = False
    print('CV_GROUP not found -> KFold fallback')
    train.drop(columns=['ID', 'TARGET'], inplace=True)

test.drop(columns=['ID'], inplace=True)

assert list(train.columns) == list(test.columns), 'FATAL: column mismatch'

print(f'Features: {train.shape[1]}')
print(f'NaN train: {train.isnull().sum().sum():,} | NaN test: {test.isnull().sum().sum():,}')
print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

Loading data...
Train: (661574, 448) | Test: (410139, 446)
Loaded in 12.6s
CV_GROUP: 70 groups -> GroupKFold
Features: 445
NaN train: 0 | NaN test: 0
Elapsed: 24s


In [19]:
# CELL 4 - TARGET ANALYSIS AND CLIPPING
# P1/P99 clip. Training only. Test unconstrained.

desc = y_raw.describe(percentiles=[0.01, 0.02, 0.05, 0.25, 0.5, 0.75, 0.95, 0.98, 0.99])
print(desc)
print(f'Skewness: {y_raw.skew():.4f} | Kurtosis: {y_raw.kurtosis():.4f}')

if CFG.TARGET_CLIP:
    clip_lo   = y_raw.quantile(CFG.CLIP_LOWER / 100)
    clip_hi   = y_raw.quantile(CFG.CLIP_UPPER / 100)
    y_series  = y_raw.clip(clip_lo, clip_hi)
    n_clipped = int(((y_raw < clip_lo) | (y_raw > clip_hi)).sum())
    print(f'Clip: [{clip_lo:.6f}, {clip_hi:.6f}] -> {n_clipped:,} rows ({100*n_clipped/len(y_raw):.2f}%)')
    print(f'Std: {y_raw.std():.6f} -> {y_series.std():.6f}')
    print(f'Kurtosis after clip: {y_series.kurtosis():.2f}')
    y = y_series.values.astype(np.float32)
else:
    y = y_raw.values.astype(np.float32)

# Huber delta: set to ~1.5 * std of clipped target
# This means ~87% of rows get quadratic loss, 13% get linear
y_std = float(np.std(y))
HUBER_DELTA = round(1.5 * y_std, 4)
print(f'Huber delta (1.5 * std): {HUBER_DELTA}')
print('Target ready OK')

count    661574.000000
mean         -0.000036
std           0.036660
min          -1.285424
1%           -0.097335
2%           -0.075900
5%           -0.051777
25%          -0.016153
50%           0.000000
75%           0.016073
95%           0.051537
98%           0.075723
99%           0.097421
max           1.346982
Name: TARGET, dtype: float64
Skewness: 0.1691 | Kurtosis: 48.0903
Clip: [-0.097335, 0.097421] -> 13,232 rows (2.00%)
Std: 0.036660 -> 0.031245
Kurtosis after clip: 1.52
Huber delta (1.5 * std): 0.0469
Target ready OK


In [20]:
# CELL 5 - FEATURE CATEGORIZATION
all_cols  = train.columns.tolist()

lag1_cols = [c for c in all_cols if c.endswith('_LagT1')]
lag2_cols = [c for c in all_cols if c.endswith('_LagT2')]
lag3_cols = [c for c in all_cols if c.endswith('_LagT3')]
lag_set   = set(lag1_cols + lag2_cols + lag3_cols)
base_cols = [c for c in all_cols if c not in lag_set]

special_cols   = ['Price', 'SO3_T']
pure_base_cols = [c for c in base_cols if c not in special_cols]

def get_family(col):
    return col.split('_')[0]

families = sorted(set(get_family(c) for c in pure_base_cols))

lag_map = {
    c: {
        'lag1': c + '_LagT1' if (c + '_LagT1') in lag_set else None,
        'lag2': c + '_LagT2' if (c + '_LagT2') in lag_set else None,
        'lag3': c + '_LagT3' if (c + '_LagT3') in lag_set else None,
    }
    for c in pure_base_cols
}

print(f'Pure base: {len(pure_base_cols)} | LagT1/T2/T3: {len(lag1_cols)}/{len(lag2_cols)}/{len(lag3_cols)}')
print(f'Families: {families}')
print(f'Total orig feats: {len(all_cols)}')

Pure base: 110 | LagT1/T2/T3: 111/111/111
Families: ['S01', 'S02', 'S03', 'S04', 'S05']
Total orig feats: 445


In [21]:
# CELL 6 - FEATURE ENGINEERING (v6 enhanced)
#
# ALL operations are row-wise. No cross-sample computation.
# Same function applied identically to train and test.
#
# NEW IN v6:
#
# F. INTRA-FAMILY Z-SCORE (cross-sectional neutralization)
#    For each feature f in family F at time t:
#    z_f = (f - mean_F) / (std_F + eps)
#    Theoretical basis: Fama-French factor model.
#    Removes common beta exposure, isolates idiosyncratic alpha.
#    Row-wise: mean_F and std_F are computed from OTHER features IN THE SAME ROW
#    (cross-feature aggregation, not cross-row) -> NO leakage.
#    Example: if feature is 0.05 but family mean is 0.04 and std is 0.01,
#    z = 1.0 -> this instrument is 1 std above its peer group.
#    Persistent above-peer behavior = trend continuation signal.
#
# G. PRICE SQRT-IMPACT FEATURES
#    Based on Almgren et al. (2003) market impact model:
#    large price moves have diminishing marginal impact on future returns.
#    Price_MomImpact = sign(LagT1) * |LagT1|^0.5
#    This transforms the momentum signal to capture:
#    - direction (sign preserved)
#    - nonlinear magnitude scaling (sqrt reduces influence of extreme moves)
#    Expected: better calibration for momentum continuation at high magnitudes.

def feature_engineering(df, lag_map, families, pure_base_cols, verbose=True):
    df = df.copy()
    new_feats = []

    # A. MOMENTUM (Acc12, Acc23, Div13)
    n0 = len(new_feats)
    for base_col, lags in lag_map.items():
        l1, l2, l3 = lags['lag1'], lags['lag2'], lags['lag3']
        if l1 and l2 and l1 in df.columns and l2 in df.columns:
            df[base_col + '_Acc12'] = df[l1] - df[l2]
            new_feats.append(base_col + '_Acc12')
        if l2 and l3 and l2 in df.columns and l3 in df.columns:
            df[base_col + '_Acc23'] = df[l2] - df[l3]
            new_feats.append(base_col + '_Acc23')
        if l1 and l3 and l1 in df.columns and l3 in df.columns:
            df[base_col + '_Div13'] = df[l1] - df[l3]
            new_feats.append(base_col + '_Div13')
    if verbose: print(f'  Momentum (Acc12/Acc23/Div13): {len(new_feats)-n0}')

    # B. ABSOLUTE MOMENTUM
    n0 = len(new_feats)
    for base_col, lags in lag_map.items():
        l1 = lags['lag1']
        if l1 and l1 in df.columns:
            df[base_col + '_AbsMom'] = df[l1].abs()
            new_feats.append(base_col + '_AbsMom')
    if verbose: print(f'  AbsMom (|LagT1|):             {len(new_feats)-n0}')

    # C. PRICE-SPECIFIC
    if 'Price' in df.columns:
        n0 = len(new_feats)
        p_l1, p_l2, p_l3 = 'Price_LagT1', 'Price_LagT2', 'Price_LagT3'
        if p_l1 in df.columns and p_l2 in df.columns:
            df['Price_Acc']    = df[p_l1] - df[p_l2]; new_feats.append('Price_Acc')
        if p_l1 in df.columns and p_l3 in df.columns:
            df['Price_Div13']  = df[p_l1] - df[p_l3]; new_feats.append('Price_Div13')
        if p_l2 in df.columns and p_l3 in df.columns:
            df['Price_Acc23']  = df[p_l2] - df[p_l3]; new_feats.append('Price_Acc23')
        if p_l1 in df.columns:
            abs_p = df[p_l1].abs()
            df['Price_AbsMom']    = abs_p
            # G. PRICE SQRT-IMPACT (Almgren et al. 2003)
            # sign(LagT1) * |LagT1|^0.5: captures diminishing marginal momentum
            df['Price_SqrtImpact'] = np.sign(df[p_l1]) * np.sqrt(abs_p)
            new_feats += ['Price_AbsMom', 'Price_SqrtImpact']
        if verbose: print(f'  Price-specific:               {len(new_feats)-n0}')

    # D. FAMILY AGGREGATES
    n0 = len(new_feats)
    family_means = {}
    family_stds  = {}
    for fam in families:
        fam_base = [c for c in pure_base_cols if get_family(c) == fam and c in df.columns]
        fam_lag1 = [c + '_LagT1' for c in fam_base if c + '_LagT1' in df.columns]
        if len(fam_base) < 2: continue
        fmean = df[fam_base].mean(axis=1)
        fstd  = df[fam_base].std(axis=1)
        df[fam + '_BaseMean'] = fmean
        df[fam + '_BaseStd']  = fstd
        family_means[fam]     = fmean
        family_stds[fam]      = fstd
        new_feats += [fam + '_BaseMean', fam + '_BaseStd']
        if len(fam_lag1) >= 2:
            fam_mom = df[fam_lag1]
            mmean   = fam_mom.mean(axis=1)
            mstd    = fam_mom.std(axis=1)
            df[fam + '_MomMean']   = mmean
            df[fam + '_MomStd']    = mstd
            df[fam + '_MomSpread'] = fam_mom.max(axis=1) - fam_mom.min(axis=1)
            new_feats += [fam + '_MomMean', fam + '_MomStd', fam + '_MomSpread']
    if verbose: print(f'  Family aggregates:            {len(new_feats)-n0}')

    # F. INTRA-FAMILY Z-SCORE (NEW in v6)
    # z_fi = (f_i - family_mean) / (family_std + eps)
    # Cross-sectional neutralization: removes common beta, isolates alpha.
    # This is a row-wise operation (aggregation across features in same row,
    # NOT across rows) -> fully compliant, no leakage.
    n0 = len(new_feats)
    eps = 1e-8
    for fam in families:
        fam_base = [c for c in pure_base_cols if get_family(c) == fam and c in df.columns]
        if fam not in family_means or len(fam_base) < 2: continue
        fmean = family_means[fam]
        fstd  = family_stds[fam]
        for col in fam_base:
            z_name = col + '_ZScore'
            df[z_name] = (df[col] - fmean) / (fstd + eps)
            new_feats.append(z_name)
    if verbose: print(f'  Intra-family z-score:         {len(new_feats)-n0}')

    # E. SO3_T POLYNOMIAL + INTERACTIONS
    if 'SO3_T' in df.columns:
        n0 = len(new_feats)
        df['SO3_T_sq'] = df['SO3_T'] ** 2;  new_feats.append('SO3_T_sq')
        df['SO3_T_cb'] = df['SO3_T'] ** 3;  new_feats.append('SO3_T_cb')
        if 'Price_LagT1' in df.columns:
            df['SO3T_x_PriceMom'] = df['SO3_T'] * df['Price_LagT1']
            new_feats.append('SO3T_x_PriceMom')
        for fam in families:
            fname = fam + '_MomMean'
            if fname in df.columns:
                df['SO3T_x_' + fam + '_Mom'] = df['SO3_T'] * df[fname]
                new_feats.append('SO3T_x_' + fam + '_Mom')
        if verbose: print(f'  SO3_T poly+interactions:      {len(new_feats)-n0}')

    if verbose:
        print(f'  New features total:           {len(new_feats)}')
        print(f'  Total features after FE:      {df.shape[1]}')
    return df, new_feats


print('Running feature engineering...')
t0 = time.time()
train_fe, new_feats = feature_engineering(train, lag_map, families, pure_base_cols, verbose=True)
test_fe,  _         = feature_engineering(test,  lag_map, families, pure_base_cols, verbose=False)
print(f'Done in {time.time()-t0:.1f}s')
print(f'Train_FE: {train_fe.shape} | Test_FE: {test_fe.shape}')
print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

Running feature engineering...
  Momentum (Acc12/Acc23/Div13): 330
  AbsMom (|LagT1|):             110
  Price-specific:               5
  Family aggregates:            20
  Intra-family z-score:         109
  SO3_T poly+interactions:      7
  New features total:           581
  Total features after FE:      1026
Done in 35.3s
Train_FE: (661574, 1026) | Test_FE: (410139, 1026)
Elapsed: 70s


In [22]:
# CELL 7 - FINALIZE ARRAYS + FEATURE NORMALIZATION
#
# v6 NEW: z-score normalization using TRAIN statistics only.
#
# Why normalize for tree models?
# Tree models are scale-invariant in theory. But LGB uses histograms:
# it discretizes each feature into B bins (default B=255).
# If feature A ranges [-1000, 1000] and feature B ranges [-0.01, 0.01],
# histogram bins for A are 1000x wider -> much coarser splits.
# A split at x=0 for feature A covers 1000 units of value;
# for feature B it covers 0.01 units.
# Normalization makes all features use histogram bins at the same resolution.
# This is especially important here because features are ANONYMOUS -
# we have no idea what scale each feature is on.
#
# LEAKAGE-FREE: we use TRAIN mean/std only.
# The same transform is applied to test using train statistics.
# This is exactly what any proper ML pipeline does (StandardScaler fit on train).

train_fe.replace([np.inf, -np.inf], np.nan, inplace=True)
test_fe.replace( [np.inf, -np.inf], np.nan, inplace=True)

assert list(train_fe.columns) == list(test_fe.columns), 'FATAL: column mismatch'

feature_cols_full = train_fe.columns.tolist()
n_features_full   = len(feature_cols_full)
print(f'Features before selection: {n_features_full}')

X_full      = train_fe.values.astype(np.float32)
X_test_full = test_fe.values.astype(np.float32)

del train_fe, test_fe
gc.collect()

# Normalization: train stats only
print('Normalizing features (train stats only)...')
col_mean = np.nanmean(X_full, axis=0, keepdims=True).astype(np.float32)
col_std  = np.nanstd( X_full, axis=0, keepdims=True).astype(np.float32)
col_std  = np.where(col_std < 1e-8, 1.0, col_std)  # avoid div by zero

X_full      = (X_full      - col_mean) / col_std
X_test_full = (X_test_full - col_mean) / col_std

# Replace any remaining NaN with 0 (after norm, 0 = mean imputation)
X_full      = np.nan_to_num(X_full,      nan=0.0, posinf=0.0, neginf=0.0)
X_test_full = np.nan_to_num(X_test_full, nan=0.0, posinf=0.0, neginf=0.0)

print(f'X_full: {X_full.shape} | X_test_full: {X_test_full.shape}')
print(f'y: {y.shape} float32')
print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

Features before selection: 1026
Normalizing features (train stats only)...
X_full: (661574, 1026) | X_test_full: (410139, 1026)
y: (661574,) float32
Elapsed: 186s


In [23]:
# CELL 8 - FEATURE SELECTION
#
# Strategy:
#   1. Train one LGB fold (fold 1) on all features
#   2. Extract feature importance (gain)
#   3. Keep top CFG.TOP_K_FEATURES by gain
#   4. Use only those features for all subsequent training
#
# Justification:
#   v5 result: LGB stopped at iter 22 with 916 features.
#   22 iters * 63 leaves = 1386 total splits across 916 features.
#   Many features were never split on even once.
#   With top 300 features: 1386 splits / 300 features = 4.6 splits/feature.
#   Better exploration of each feature's predictive content.
#
# This is a ONE-TIME operation using fold 1 only.
# Cost: ~1 fold * ~60s = ~1 min.

print('CV setup...')
if USE_GROUP_CV:
    splitter = GroupKFold(n_splits=CFG.N_FOLDS)
    splits   = list(splitter.split(X_full, y, groups=cv_groups))
    print(f'GroupKFold: {n_groups} groups -> {CFG.N_FOLDS} folds')
else:
    kf     = KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    splits = list(kf.split(X_full, y))
    print('KFold fallback')

print(f'Running feature selection (fold 1, LGB gain importance)...')
t0 = time.time()

trn_fs, val_fs = splits[0]

fs_params = {
    **CFG.LGB_PARAMS,
    'learning_rate': 0.05,   # faster for selection run only
    'num_leaves':    31,     # smaller for speed
}

d_fs_trn = lgb.Dataset(X_full[trn_fs], label=y[trn_fs],
                        feature_name=feature_cols_full, free_raw_data=True)
d_fs_val = lgb.Dataset(X_full[val_fs], label=y[val_fs],
                        reference=d_fs_trn, free_raw_data=True)

fs_model = lgb.train(
    fs_params,
    d_fs_trn,
    num_boost_round=500,
    valid_sets=[d_fs_val],
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(-1),
    ],
)

# Feature importance by gain (not splits)
# Gain = total reduction in loss attributed to each feature
# More meaningful than split count for identifying true signal features
importance = fs_model.feature_importance(importance_type='gain')
fi_series  = pd.Series(importance, index=feature_cols_full).sort_values(ascending=False)

# Keep top K features with strictly positive importance
positive_feats = fi_series[fi_series > 0]
top_k          = min(CFG.TOP_K_FEATURES, len(positive_feats))
selected_feats = positive_feats.head(top_k).index.tolist()

zero_imp = int((importance == 0).sum())
print(f'Feature selection done in {time.time()-t0:.1f}s')
print(f'Zero importance features:  {zero_imp}/{n_features_full}')
print(f'Positive importance:       {len(positive_feats)}')
print(f'Selected (top {top_k}):       {len(selected_feats)}')
print(f'Top 10 features:')
print(fi_series.head(10).to_string())

# Save importance
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fi_series.reset_index().rename(columns={'index': 'feature', 0: 'gain'}).to_csv(
    CFG.OUTPUT_DIR / 'feature_importance.csv', index=False)

# Subset arrays to selected features
sel_idx    = [feature_cols_full.index(f) for f in selected_feats]
X          = X_full[:, sel_idx]
X_test     = X_test_full[:, sel_idx]
feature_cols = selected_feats
n_features   = len(feature_cols)

del fs_model, d_fs_trn, d_fs_val, X_full, X_test_full
gc.collect()

print(f'Final arrays: X={X.shape} | X_test={X_test.shape}')
print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

CV setup...
GroupKFold: 70 groups -> 5 folds
Running feature selection (fold 1, LGB gain importance)...
Feature selection done in 48.8s
Zero importance features:  519/1026
Positive importance:       507
Selected (top 300):       300
Top 10 features:
S03_P01_D01_S02_Acc23                1.015276
S03_D02_A09_A02_B03_E03_E04_LagT1    0.857638
S03_A02_A04_D04_F04_U03_ZScore       0.808283
S03_V14_V01                          0.721142
S03_P01_D01_S01_Acc12                0.688065
S03_V04_T03_LagT2                    0.670517
S03_D02_V01_A01_B10_E10_E11          0.651859
S03_D02_A09_A02_B03_E03_E04_LagT2    0.641998
S03_D02_A09_A02_B03_E03_E04          0.622600
S03_A07_A05_V09_LagT3                0.620198
Final arrays: X=(661574, 300) | X_test=(410139, 300)
Elapsed: 239s


In [24]:
# CELL 9 - OVERFIT DIAGNOSTIC
# Quick fold 1 check with final params on selected features.

print('OVERFIT DIAGNOSTIC (fold 1, Huber LGB)')
print('-' * 50)

trn_d, val_d = splits[0]

dtrain_d = lgb.Dataset(X[trn_d], label=y[trn_d], feature_name=feature_cols, free_raw_data=True)
dval_d   = lgb.Dataset(X[val_d], label=y[val_d], reference=dtrain_d, free_raw_data=True)

diag_model = lgb.train(
    CFG.LGB_PARAMS,
    dtrain_d,
    num_boost_round=CFG.LGB_NUM_ROUNDS,
    valid_sets=[dtrain_d, dval_d],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(CFG.LGB_EARLY_STOP, verbose=False),
        lgb.log_evaluation(500),
    ],
)

best_it       = diag_model.best_iteration
train_r2_diag = r2_score(y[trn_d], diag_model.predict(X[trn_d], num_iteration=best_it))
val_r2_diag   = r2_score(y[val_d], diag_model.predict(X[val_d], num_iteration=best_it))
gap           = train_r2_diag - val_r2_diag

print(f'Features used: {n_features}')
print(f'Best iteration: {best_it}')
print(f'Train R2: {train_r2_diag:.6f}')
print(f'Val   R2: {val_r2_diag:.6f}')
print(f'Gap:      {gap:.6f}', end='  ')

if gap < 0.005:   print('EXCELLENT')
elif gap < 0.015: print('GOOD')
elif gap < 0.03:  print('MODERATE')
else:             print('HIGH - increase regularization')

# KEY DIAGNOSTIC: check if model is still underfitting
if best_it < 100:
    print(f'WARNING: model stopped at {best_it} iter - still underfitting')
    print('  Consider: lower lambda_l2, fewer min_child_samples, lower min_gain_to_split')
elif best_it > 2800:
    print(f'NOTE: model hit max rounds ({best_it}) - consider increasing LGB_NUM_ROUNDS')
else:
    print(f'GOOD: model converged at {best_it} iter (healthy range 100-2800)')

print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

del diag_model, dtrain_d, dval_d
gc.collect()

OVERFIT DIAGNOSTIC (fold 1, Huber LGB)
--------------------------------------------------
Features used: 300
Best iteration: 81
Train R2: 0.015953
Val   R2: 0.000107
Gap:      0.015846  MODERATE
  Consider: lower lambda_l2, fewer min_child_samples, lower min_gain_to_split
Elapsed: 294s


982

In [25]:
# CELL 10 - LGB HUBER: 5-FOLD CV
# Objective: huber (alpha=0.9)
# Robust to fat-tailed target (kurtosis=48)

oof_lgb    = np.zeros(len(y), dtype=np.float64)
pred_lgb   = np.zeros(len(X_test), dtype=np.float64)
lgb_scores = []
lgb_iters  = []

print('=' * 50)
print(f'LGB Huber - {CFG.N_FOLDS}-Fold CV  ({n_features} features)')
print('=' * 50)

for fold, (trn_idx, val_idx) in enumerate(splits):
    elapsed = time.time() - NOTEBOOK_START
    if elapsed > CFG.RUNTIME_LIMIT_S * 0.30 and fold >= 3:
        print(f'Time budget risk ({elapsed:.0f}s). Stopping LGB Huber after {fold} folds.')
        break

    t0 = time.time()
    dtrain = lgb.Dataset(X[trn_idx], label=y[trn_idx], feature_name=feature_cols, free_raw_data=True)
    dval   = lgb.Dataset(X[val_idx], label=y[val_idx], reference=dtrain, free_raw_data=True)

    model = lgb.train(
        CFG.LGB_PARAMS,
        dtrain,
        num_boost_round=CFG.LGB_NUM_ROUNDS,
        valid_sets=[dtrain, dval],
        valid_names=['train', 'valid'],
        callbacks=[
            lgb.early_stopping(CFG.LGB_EARLY_STOP, verbose=False),
            lgb.log_evaluation(500),
        ],
    )

    best_it          = model.best_iteration
    lgb_iters.append(best_it)
    val_preds        = model.predict(X[val_idx], num_iteration=best_it)
    oof_lgb[val_idx] = val_preds
    pred_lgb        += model.predict(X_test, num_iteration=best_it) / CFG.N_FOLDS

    fold_r2 = r2_score(y[val_idx], val_preds)
    lgb_scores.append(fold_r2)
    print(f'  Fold {fold+1}/{CFG.N_FOLDS} | R2: {fold_r2:+.6f} | iter: {best_it} | {time.time()-t0:.0f}s')

    del model, dtrain, dval
    gc.collect()

n_lgb_folds = len(lgb_scores)
if n_lgb_folds < CFG.N_FOLDS:
    print(f'Only {n_lgb_folds}/{CFG.N_FOLDS} folds completed')
    pred_lgb = pred_lgb * CFG.N_FOLDS / n_lgb_folds
    filled   = np.concatenate([splits[i][1] for i in range(n_lgb_folds)])
    lgb_oof_r2 = r2_score(y[filled], oof_lgb[filled])
else:
    lgb_oof_r2 = r2_score(y, oof_lgb)

print(f'LGB Huber OOF R2: {lgb_oof_r2:+.6f} | fold std: {np.std(lgb_scores):.6f} | avg iter: {np.mean(lgb_iters):.0f}')
print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

LGB Huber - 5-Fold CV  (300 features)
  Fold 1/5 | R2: +0.000107 | iter: 81 | 48s
[500]	train's huber: 0.000455188	valid's huber: 0.000470017
[1000]	train's huber: 0.000429663	valid's huber: 0.000470079
  Fold 2/5 | R2: +0.002741 | iter: 794 | 114s
  Fold 3/5 | R2: +0.001350 | iter: 178 | 61s
Time budget risk (527s). Stopping LGB Huber after 3 folds.
Only 3/5 folds completed
LGB Huber OOF R2: +0.001448 | fold std: 0.001076 | avg iter: 351
Elapsed: 527s


In [26]:
# CELL 11 - LGB QUANTILE (alpha=0.5): 5-FOLD CV
#
# Median regression = L1 loss minimization.
# Statistical properties:
#   - Breakdown point = 50%: up to 50% of data can be outliers without bias
#   - Optimal under Laplace (double-exponential) error distribution
#   - Financial returns are well-approximated by Laplace (heavier than Gaussian)
#   - Different gradient signal than Huber -> decorrelated predictions
# Ensemble value:
#   Var[avg(pred1, pred2)] = 0.5*Var[pred1] + 0.5*Var[pred2] + Cov[pred1,pred2]
#   Lower Cov (decorrelated) -> lower ensemble variance -> better generalization

oof_lgbq    = np.zeros(len(y), dtype=np.float64)
pred_lgbq   = np.zeros(len(X_test), dtype=np.float64)
lgbq_scores = []
lgbq_iters  = []

print('=' * 50)
print(f'LGB Quantile(0.5) - {CFG.N_FOLDS}-Fold CV  ({n_features} features)')
print('=' * 50)

for fold, (trn_idx, val_idx) in enumerate(splits):
    elapsed = time.time() - NOTEBOOK_START
    if elapsed > CFG.RUNTIME_LIMIT_S * 0.55 and fold >= 3:
        print(f'Time budget risk ({elapsed:.0f}s). Stopping LGB Quantile after {fold} folds.')
        break

    t0 = time.time()
    dtrain = lgb.Dataset(X[trn_idx], label=y[trn_idx], feature_name=feature_cols, free_raw_data=True)
    dval   = lgb.Dataset(X[val_idx], label=y[val_idx], reference=dtrain, free_raw_data=True)

    model = lgb.train(
        CFG.LGB_Q_PARAMS,
        dtrain,
        num_boost_round=CFG.LGB_Q_NUM_ROUNDS,
        valid_sets=[dtrain, dval],
        valid_names=['train', 'valid'],
        callbacks=[
            lgb.early_stopping(CFG.LGB_Q_EARLY_STOP, verbose=False),
            lgb.log_evaluation(500),
        ],
    )

    best_it             = model.best_iteration
    lgbq_iters.append(best_it)
    val_preds           = model.predict(X[val_idx], num_iteration=best_it)
    oof_lgbq[val_idx]   = val_preds
    pred_lgbq          += model.predict(X_test, num_iteration=best_it) / CFG.N_FOLDS

    fold_r2 = r2_score(y[val_idx], val_preds)
    lgbq_scores.append(fold_r2)
    print(f'  Fold {fold+1}/{CFG.N_FOLDS} | R2: {fold_r2:+.6f} | iter: {best_it} | {time.time()-t0:.0f}s')

    del model, dtrain, dval
    gc.collect()

n_lgbq_folds = len(lgbq_scores)
if n_lgbq_folds < CFG.N_FOLDS:
    print(f'Only {n_lgbq_folds}/{CFG.N_FOLDS} folds completed')
    pred_lgbq = pred_lgbq * CFG.N_FOLDS / n_lgbq_folds
    filled    = np.concatenate([splits[i][1] for i in range(n_lgbq_folds)])
    lgbq_oof_r2 = r2_score(y[filled], oof_lgbq[filled])
else:
    lgbq_oof_r2 = r2_score(y, oof_lgbq)

print(f'LGB Quantile OOF R2: {lgbq_oof_r2:+.6f} | fold std: {np.std(lgbq_scores):.6f} | avg iter: {np.mean(lgbq_iters):.0f}')
print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

LGB Quantile(0.5) - 5-Fold CV  (300 features)
[500]	train's quantile: 0.0110286	valid's quantile: 0.0110838
  Fold 1/5 | R2: -0.000854 | iter: 370 | 117s
[500]	train's quantile: 0.0110205	valid's quantile: 0.0111223
  Fold 2/5 | R2: +0.000503 | iter: 501 | 153s
[500]	train's quantile: 0.0109239	valid's quantile: 0.0115054
[1000]	train's quantile: 0.0106917	valid's quantile: 0.0115059
  Fold 3/5 | R2: +0.001292 | iter: 722 | 150s
Time budget risk (948s). Stopping LGB Quantile after 3 folds.
Only 3/5 folds completed
LGB Quantile OOF R2: +0.000384 | fold std: 0.000886 | avg iter: 531
Elapsed: 948s


In [27]:
# CELL 12 - CATBOOST HUBER: 5-FOLD CV (PRIMARY MODEL)
#
# CatBoost structural advantages remain after fixing CV:
#   - Ordered boosting: no within-fold gradient leakage
#   - Oblivious (symmetric) trees: shared split per level = strong L1 regularization
#   - Better on low-SNR noisy targets
# Huber loss: loss_function='Huber:delta=0.05'
#   delta=0.05 ~= 1.5 * std of clipped target
#   -> 87% of rows quadratic loss, 13% linear

oof_cat    = np.zeros(len(y), dtype=np.float64)
pred_cat   = np.zeros(len(X_test), dtype=np.float64)
cat_scores = []
cat_iters  = []
CATBOOST_DONE = False

if CATBOOST_AVAILABLE:
    print('=' * 50)
    print(f'CatBoost Huber - {CFG.N_FOLDS}-Fold CV  ({n_features} features)')
    print('=' * 50)

    for fold, (trn_idx, val_idx) in enumerate(splits):
        elapsed   = time.time() - NOTEBOOK_START
        remaining = CFG.RUNTIME_LIMIT_S - elapsed
        folds_left = CFG.N_FOLDS - fold
        print(f'  Fold {fold+1} | {remaining:.0f}s remaining')

        if remaining < 60:
            print('TIME BUDGET EXHAUSTED - stopping CatBoost')
            break

        iters_this_fold = CFG.CAT_PARAMS['iterations']
        if folds_left > 0:
            iters_budget    = int((remaining - 60) / folds_left / CFG.CAT_SECS_PER_ITER)
            iters_this_fold = min(iters_this_fold, max(400, iters_budget))
            if iters_this_fold < CFG.CAT_PARAMS['iterations']:
                print(f'  Reducing to {iters_this_fold} iter (time constraint)')

        cat_params_fold = {**CFG.CAT_PARAMS, 'iterations': iters_this_fold}

        t0 = time.time()
        pool_trn = Pool(X[trn_idx], y[trn_idx])
        pool_val = Pool(X[val_idx], y[val_idx])

        model = CatBoostRegressor(**cat_params_fold)
        model.fit(pool_trn, eval_set=pool_val, use_best_model=True)

        val_preds        = model.predict(X[val_idx])
        oof_cat[val_idx] = val_preds
        pred_cat        += model.predict(X_test) / CFG.N_FOLDS

        fold_r2 = r2_score(y[val_idx], val_preds)
        best_it = model.get_best_iteration()
        cat_scores.append(fold_r2)
        cat_iters.append(best_it)
        print(f'  Fold {fold+1}/{CFG.N_FOLDS} | R2: {fold_r2:+.6f} | best_iter: {best_it} | {time.time()-t0:.0f}s')

        del model, pool_trn, pool_val
        gc.collect()

    n_cat_folds = len(cat_scores)
    if n_cat_folds == 0:
        print('No CatBoost folds completed — skipping safely')
        cat_oof_r2 = 0.0
        cat_std = 0.0
        pred_cat = np.zeros(len(X_test))
        CATBOOST_DONE = False
    elif n_cat_folds < CFG.N_FOLDS:
        print(f'Only {n_cat_folds}/{CFG.N_FOLDS} folds completed')
        pred_cat = pred_cat * CFG.N_FOLDS / n_cat_folds
        filled = np.concatenate([splits[i][1] for i in range(n_cat_folds)])
        cat_oof_r2 = r2_score(y[filled], oof_cat[filled])
    else:
        cat_oof_r2 = r2_score(y, oof_cat)
    cat_std       = np.std(cat_scores)
    CATBOOST_DONE = True

    print(f'CatBoost OOF R2: {cat_oof_r2:+.6f} | fold std: {cat_std:.6f} | avg iter: {np.mean(cat_iters):.0f}')
    print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s')
else:
    cat_oof_r2 = 0.0
    cat_std    = 0.0
    print('CatBoost not available')

CatBoost Huber - 5-Fold CV  (300 features)
  Fold 1 | 659s remaining
  Reducing to 518 iter (time constraint)
0:	learn: 0.0313668	test: 0.0307398	best: 0.0307398 (0)	total: 332ms	remaining: 2m 51s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.03073411616
bestIteration = 76

Shrink model to first 77 iterations.
  Fold 1/5 | R2: +0.000324 | best_iter: 76 | 26s
  Fold 2 | 633s remaining
  Reducing to 619 iter (time constraint)
0:	learn: 0.0313783	test: 0.0306990	best: 0.0306990 (0)	total: 187ms	remaining: 1m 55s
200:	learn: 0.0310496	test: 0.0306841	best: 0.0306841 (200)	total: 26.9s	remaining: 55.9s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.03068338141
bestIteration = 249

Shrink model to first 250 iterations.
  Fold 2/5 | R2: +0.001009 | best_iter: 249 | 52s
  Fold 3 | 581s remaining
  Reducing to 751 iter (time constraint)
0:	learn: 0.0311254	test: 0.0317117	best: 0.0317117 (0)	total: 188ms	remaining: 2m 20s
Stopped by overfitting detec

In [28]:
# CELL 13 - ENSEMBLE OPTIMIZATION
# Three models: LGB Huber + LGB Quantile + CatBoost Huber
# OOF-based softmax weight optimization. No test exploitation.
#
# Correlation check: if two models have corr > 0.95,
# one is redundant and gets low weight automatically.

model_names = ['LGB_Huber', 'LGB_Q50']
oofs_list   = [oof_lgb, oof_lgbq]
preds_list  = [pred_lgb, pred_lgbq]

if CATBOOST_DONE:
    model_names.append('CAT_Huber')
    oofs_list.append(oof_cat)
    preds_list.append(pred_cat)

print('MODEL OOF R2 AND PAIRWISE CORRELATIONS')
print('-' * 50)
for name, oof in zip(model_names, oofs_list):
    print(f'  {name:<12} OOF R2: {r2_score(y, oof):+.6f}')

print('\nPairwise OOF correlations:')
for i in range(len(oofs_list)):
    for j in range(i+1, len(oofs_list)):
        corr = np.corrcoef(oofs_list[i], oofs_list[j])[0,1]
        print(f'  {model_names[i]} vs {model_names[j]}: {corr:.4f}')

oof_sa  = np.mean(oofs_list, axis=0)
pred_sa = np.mean(preds_list, axis=0)
r2_sa   = r2_score(y, oof_sa)
print(f'\nSimple avg OOF R2: {r2_sa:+.6f}')

if len(oofs_list) > 1:
    oofs_arr  = np.stack(oofs_list, axis=1)
    preds_arr = np.stack(preds_list, axis=1)

    def neg_r2(raw_w):
        w     = np.exp(raw_w) / np.exp(raw_w).sum()
        blend = (oofs_arr * w).sum(axis=1)
        return -r2_score(y, blend)

    best_r2_opt, best_w = -np.inf, None
    for seed in range(15):
        rng    = np.random.RandomState(seed)
        w0     = rng.randn(len(oofs_list)) * 0.1
        result = minimize(neg_r2, w0, method='Nelder-Mead',
                          options={'maxiter': 3000, 'xatol': 1e-7, 'fatol': 1e-9})
        if -result.fun > best_r2_opt:
            best_r2_opt = -result.fun
            best_w      = np.exp(result.x) / np.exp(result.x).sum()

    oof_opt  = (oofs_arr  * best_w).sum(axis=1)
    pred_opt = (preds_arr * best_w).sum(axis=1)
    r2_opt   = r2_score(y, oof_opt)

    print(f'Optimized OOF R2:  {r2_opt:+.6f}')
    for name, w in zip(model_names, best_w):
        print(f'  Weight {name:<12}: {w:.4f}')

    if r2_opt > r2_sa:
        final_oof, final_pred = oof_opt, pred_opt
        final_r2, method_str  = r2_opt, 'Optimized wts'
        print(f'Using OPTIMIZED weights (+{r2_opt-r2_sa:.6f})')
    else:
        final_oof, final_pred = oof_sa, pred_sa
        final_r2, method_str  = r2_sa, 'Simple avg'
        print('Using SIMPLE AVERAGE')
else:
    final_oof  = oofs_list[0]
    final_pred = preds_list[0]
    final_r2   = r2_score(y, final_oof)
    method_str = model_names[0]
    r2_opt     = final_r2

print(f'FINAL OOF R2: {final_r2:+.6f}')
print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

MODEL OOF R2 AND PAIRWISE CORRELATIONS
--------------------------------------------------
  LGB_Huber    OOF R2: +0.000856
  LGB_Q50      OOF R2: +0.000227
  CAT_Huber    OOF R2: +0.000986

Pairwise OOF correlations:
  LGB_Huber vs LGB_Q50: 0.5628
  LGB_Huber vs CAT_Huber: 0.4475
  LGB_Q50 vs CAT_Huber: 0.5161

Simple avg OOF R2: +0.001431
Optimized OOF R2:  +0.001476
  Weight LGB_Huber   : 0.3546
  Weight LGB_Q50     : 0.1917
  Weight CAT_Huber   : 0.4536
Using OPTIMIZED weights (+0.000045)
FINAL OOF R2: +0.001476
Elapsed: 1270s


In [29]:
# CELL 14 - SUBMISSION GUARDS AND SAVE

CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

submission = pd.DataFrame({'ID': test_ids, 'TARGET': final_pred})

assert len(submission)            == len(test_ids),             'FATAL: Wrong row count!'
assert list(submission.columns)  == ['ID', 'TARGET'],          'FATAL: Wrong columns!'
assert submission['ID'].nunique() == len(test_ids),            'FATAL: Duplicate IDs!'
assert not submission['TARGET'].isnull().any(),                 'FATAL: NaN in TARGET!'
assert not np.isinf(submission['TARGET'].values).any(),         'FATAL: Inf in TARGET!'
assert submission['TARGET'].std() > 0,                          'FATAL: Constant predictions!'

print('All 6 guards passed OK')

sub_path = CFG.OUTPUT_DIR / 'submission.csv'
submission.to_csv(sub_path, index=False)

verify = pd.read_csv(sub_path)
assert len(verify) == len(test_ids)
assert list(verify.columns) == ['ID', 'TARGET']

total_elapsed = time.time() - NOTEBOOK_START
print(f'submission.csv SAVED AND VERIFIED')
print(f'  Rows:     {len(submission):,}')
print(f'  ID range: {submission["ID"].min()} to {submission["ID"].max()}')
print(f'\nTARGET stats:')
print(submission['TARGET'].describe())
print(f'\nTotal elapsed: {total_elapsed:.0f}s = {total_elapsed/60:.1f} min')

All 6 guards passed OK
submission.csv SAVED AND VERIFIED
  Rows:     410,139
  ID range: 672374 to 1089244

TARGET stats:
count    410139.000000
mean         -0.000004
std           0.001380
min          -0.013478
25%          -0.000720
50%           0.000024
75%           0.000750
max           0.014576
Name: TARGET, dtype: float64

Total elapsed: 1276s = 21.3 min


In [30]:
# CELL 15 - FINAL SCORECARD

total_elapsed = time.time() - NOTEBOOK_START

print()
print('=' * 70)
print('  R2 SCORECARD - iRage Short-Horizon Return Prediction v6')
print('=' * 70)
print(f'  Features orig -> selected: {n_features_full} -> {n_features}')
print(f'  Train: {len(X):,} | Test: {len(X_test):,}')
print(f'  CV: {"GroupKFold(CV_GROUP)" if USE_GROUP_CV else "KFold"}')
print(f'  Runtime: {total_elapsed:.0f}s = {total_elapsed/60:.1f} min')
print('-' * 70)
print('  MODEL OOF R2')
for name, oof in zip(model_names, oofs_list):
    r2  = r2_score(y, oof)
    bar = '#' * min(40, max(0, int(r2 * 5000)))
    print(f'  {name:<14} {r2:+.6f}  {bar}')
print('-' * 70)
print('  ENSEMBLE')
print(f'  Simple avg:    {r2_sa:+.6f}')
if len(oofs_list) > 1: print(f'  Optimized:     {r2_opt:+.6f}')
print(f'  FINAL ({method_str}): {final_r2:+.6f}')
print('-' * 70)
print('  OVERFIT CHECK (LGB Huber fold 1)')
print(f'  Train R2: {train_r2_diag:.6f} | Val R2: {val_r2_diag:.6f} | Gap: {gap:.6f}')
print(f'  {"LOW OVERFIT" if gap<0.015 else "CHECK OVERFIT"}')
print('-' * 70)
print('  v6 CHANGES (quant-justified)')
print('  + LGB LR=0.01, num_leaves=63 (fix underfitting - was stopping @22 iter)')
print('  + Huber loss (kurtosis=48 -> MSE blown by outliers)')
print('  + Quantile(0.5) model (L1 optimal under Laplace error distribution)')
print('  + Feature selection top 300 by gain (removed noise features)')
print('  + Intra-family z-score (Fama-French cross-sectional neutralization)')
print('  + Feature normalization (histogram bin resolution equalization)')
print('  + Price sqrt-impact feature (Almgren et al. 2003)')
print('  - Rejected GPT group-normalization (CV_GROUP absent from test set)')
print('-' * 70)
print('  COMPLIANCE')
print('  OK Data: /kaggle/input/competitions/...')
print('  OK No internet / pip / external data')
print('  OK GroupKFold on CV_GROUP (regime-aware CV)')
print('  OK Zero cross-sample test operations')
print('  OK Feature normalization uses train stats only (no test leakage)')
print(f'  {"OK" if total_elapsed<1800 else "FAIL"} Runtime {total_elapsed/60:.1f} min')
print('=' * 70)
print()

if final_r2 > 0.01:   print('STRONG signal - top leaderboard')
elif final_r2 > 0.003: print('GOOD signal - competitive range')
elif final_r2 > 0.001: print('MODERATE - above baseline')
elif final_r2 > 0:     print('WEAK - positive signal')
else:                   print('NO SIGNAL - revisit model')

print('\nNOTEBOOK COMPLETE v6')


  R2 SCORECARD - iRage Short-Horizon Return Prediction v6
  Features orig -> selected: 1026 -> 300
  Train: 661,574 | Test: 410,139
  CV: GroupKFold(CV_GROUP)
  Runtime: 1279s = 21.3 min
----------------------------------------------------------------------
  MODEL OOF R2
  LGB_Huber      +0.000856  ####
  LGB_Q50        +0.000227  #
  CAT_Huber      +0.000986  ####
----------------------------------------------------------------------
  ENSEMBLE
  Simple avg:    +0.001431
  Optimized:     +0.001476
  FINAL (Optimized wts): +0.001476
----------------------------------------------------------------------
  OVERFIT CHECK (LGB Huber fold 1)
  Train R2: 0.015953 | Val R2: 0.000107 | Gap: 0.015846
  CHECK OVERFIT
----------------------------------------------------------------------
  v6 CHANGES (quant-justified)
  + LGB LR=0.01, num_leaves=63 (fix underfitting - was stopping @22 iter)
  + Huber loss (kurtosis=48 -> MSE blown by outliers)
  + Quantile(0.5) model (L1 optimal under Laplace e

---
## ONE-SHOT SUBMISSION CHECKLIST v6

- [ ] Total runtime < 30 min
- [ ] LGB Huber best_iter > 100 (if still <50, model is underfitting - lower min_gain_to_split)
- [ ] LGB Huber best_iter < 2800 (if hitting max, increase LGB_NUM_ROUNDS)
- [ ] Feature selection kept > 100 features (if <100, lower TOP_K_FEATURES threshold)
- [ ] submission.csv SAVED AND VERIFIED (410,139 rows, columns ID + TARGET)
- [ ] TARGET std > 0 (non-constant predictions)
- [ ] No NaN or Inf in TARGET

## What to Try Next If R2 Still <0.001

The real signal in this problem is small. GroupKFold gives an honest estimate.
If v6 R2 is still near zero:

1. Inspect which features have highest importance - those are your signal
2. Look at families: is S03 dominant? If so, build more S03-specific features
3. Try TOP_K_FEATURES=100 (even more aggressive selection)
4. Try lambda_l2=1.0 (less regularization, more learning)
5. The target has median=0 and kurtosis=48 - there is very little signal
   in most rows. A model that correctly predicts the DIRECTION of large moves
   will have a huge advantage. Consider: directional accuracy on |y|>0.05.